## LightGBM LTV

## Идея

Ноутбук повторяет подход из `02_model_ltv`, но использует **LightGBM** вместо CatBoost.

Логика та же:
- строим признаки по временным окнам 7/14/30/60/90 дней;
- добавляем агрегаты за всю историю и признаки recency;
- делаем time-based CV по 4 anchor-датам;
- обучаем модель на `log1p(GMV)` и считаем RMSLE.

Дополнительно здесь появляются **признаки интервалов между покупками** —
средний/медианный/последний интервал и их соотношения. Это должно помочь
модели отличать «регулярных» покупателей от случайных.

## Ожидаемый результат

- OOF-оценка RMSLE для LightGBM;
- финальная модель, сохранённая в файл;
- `submission_lgbm_v1.csv` со столбцами `user_id` и `predict`.

In [ ]:
!pip install -q lightgbm pyarrow polars scikit-learn

import gc
import shutil
from pathlib import Path
from datetime import timedelta

import numpy as np
import polars as pl
import lightgbm as lgb

from sklearn.metrics import mean_squared_error

from google.colab import drive
drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/e-cup")
DATA_PATH = BASE_DIR / "data/train.parquet"

data = pl.read_parquet(DATA_PATH)

print(f"Rows: {data.height:,}")
print(f"Columns: {data.width}")
print(f"Users: {data['user_id'].n_unique():,}")
print(f"Period: {data['event_date'].min()} to {data['event_date'].max()}")


Mounted at /content/drive
Rows: 30,631,006
Columns: 18
Users: 250,000
Period: 2025-01-01 to 2026-02-13


## Настройки расчёта

Горизонт прогноза — 30 дней, окна признаков — 7/14/30/60/90 дней.
Пользователи обрабатываются батчами по 50 000, чтобы не держать все
промежуточные таблицы в памяти сразу.

Каталоги:
- `data/v4_lgbm/features` — сюда складываем признаки по фолдам;
- `data/v4_lgbm/models` — сюда сохраняем финальную модель.

In [ ]:
HORIZON = 30
N_FOLDS = 4
BATCH_SIZE = 50_000

WINDOWS = [
    ("7d", 6, 0),
    ("14d", 13, 0),
    ("30d", 29, 0),
    ("60d", 59, 0),
    ("90d", 89, 0),
]

SUM_COLS = [
    "search",
    "cat",
    "has_search_to_cart",
    "has_search_to_ord",
    "has_cat_to_cart",
    "has_cat_to_ord",
    "search_to_cart",
    "search_to_ord",
    "cat_to_cart",
    "cat_to_ord",
    "gmv_search",
    "gmv_cat",
    "to_cart",
    "to_ord",
    "gmv",
    "searches",
]

MAX_COLS = [
    "gmv",
    "to_ord",
    "to_cart",
    "searches",
]

FEATURES_DIR = BASE_DIR / "data/v4_lgbm/features"
MODELS_DIR = BASE_DIR / "data/v4_lgbm/models"
SUBMISSION_PATH = BASE_DIR / "data/submission_lgbm_v1.csv"

for path in [FEATURES_DIR, MODELS_DIR]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

user_ids = data["user_id"].unique().sort().to_list()
n_users = len(user_ids)
n_batches = (n_users + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Users: {n_users:,}")
print(f"Batches: {n_batches}")


Users: 250,000
Batches: 5


## Временные фолды и метрика

Используем ту же схему валидации, что и в CatBoost-ноутбуке:
- anchor-даты идут с шагом 14 дней;
- последние 4 anchor'а — это фолды CV;
- финальный anchor — последний день данных (2026-02-13).

Метрика — RMSLE в лог-пространстве, как в соревновании.
Target обучаем в `log1p`, чтобы модель лучше работала с нулями и длинным хвостом.

In [ ]:
def rmsle(y_true, y_pred):
    y_true = np.clip(np.asarray(y_true, dtype=np.float64), 0, None)
    y_pred = np.clip(np.asarray(y_pred, dtype=np.float64), 0, None)
    return np.sqrt(
        mean_squared_error(
            np.log1p(y_true),
            np.log1p(y_pred)
        )
    )


def generate_cv_anchor_dates(
    data,
    prediction_horizon_days=30,
    stride_days=14,
    min_history_days=90,
    n_folds=4,
):
    min_date = data["event_date"].min()
    max_date = data["event_date"].max()
    latest_anchor = max_date - timedelta(days=prediction_horizon_days)
    earliest_anchor = min_date + timedelta(days=min_history_days - 1)
    n_steps = (latest_anchor - earliest_anchor).days // stride_days

    all_anchors = [
        latest_anchor - timedelta(days=i * stride_days)
        for i in range(n_steps + 1)
    ]

    return sorted(all_anchors)[-n_folds:]


anchors = generate_cv_anchor_dates(
    data,
    prediction_horizon_days=HORIZON,
    stride_days=14,
    min_history_days=90,
    n_folds=N_FOLDS,
)

anchor_end = data["event_date"].max()

print(f"CV anchors: {anchors}")
print(f"Final anchor: {anchor_end}")


CV anchors: [datetime.date(2025, 12, 3), datetime.date(2025, 12, 17), datetime.date(2025, 12, 31), datetime.date(2026, 1, 14)]
Final anchor: 2026-02-13


## Агрегаты по окнам и за всю историю

Логика признаков совпадает с CatBoost-версией:

- по окнам 7/14/30/60/90 считаем суммы по активности, заказам, корзине, поиску и GMV;
- отдельно фиксируем количество активных дней и максимумы по ключевым колонкам;
- за всю историю считаем lifetime-агрегаты и даты последней активности,
  последнего заказа, корзины и поиска.

In [ ]:
def window_agg_exprs(anchor, windows):
    exprs = []

    for window_name, start_offset, end_offset in windows:
        window_start = anchor - timedelta(days=start_offset)
        window_end = anchor - timedelta(days=end_offset)

        mask = pl.col("event_date").is_between(
            window_start,
            window_end
        )

        for col in SUM_COLS:
            exprs.append(
                pl.when(mask)
                .then(pl.col(col))
                .otherwise(0)
                .sum()
                .alias(f"{col}_{window_name}")
            )

        exprs.append(
            pl.when(mask)
            .then(1)
            .otherwise(0)
            .sum()
            .alias(f"active_days_{window_name}")
        )

        for col in MAX_COLS:
            exprs.append(
                pl.when(mask)
                .then(pl.col(col))
                .otherwise(None)
                .max()
                .fill_null(0)
                .alias(f"max_{col}_{window_name}")
            )

    return exprs


def history_agg_exprs():
    exprs = [
        pl.len().alias("lifetime_active_days"),
        pl.col("event_date").min().alias("first_activity_date"),
        pl.col("event_date").max().alias("last_activity_date"),
        pl.when(pl.col("to_ord") > 0)
        .then(pl.col("event_date"))
        .otherwise(None)
        .max()
        .alias("last_order_date"),
        pl.when(pl.col("to_cart") > 0)
        .then(pl.col("event_date"))
        .otherwise(None)
        .max()
        .alias("last_cart_date"),
        pl.when(pl.col("searches") > 0)
        .then(pl.col("event_date"))
        .otherwise(None)
        .max()
        .alias("last_search_date"),
    ]

    for col in [
        "gmv",
        "gmv_search",
        "gmv_cat",
        "to_ord",
        "to_cart",
        "searches",
        "search_to_ord",
        "cat_to_ord",
        "search_to_cart",
        "cat_to_cart",
    ]:
        exprs.append(
            pl.col(col).sum().alias(f"lifetime_{col}")
        )

    return exprs


## Признаки интервалов между покупками

Здесь появляется то, чего не было в CatBoost-версии.

Для каждого пользователя смотрим даты его заказов и считаем:
- средний интервал между покупками;
- медианный интервал;
- std, min, max;
- средний интервал по последним 3 покупкам.

Идея: два пользователя могут иметь одинаковый GMV за 30 дней, но один покупает
раз в неделю стабильно, а второй — один раз в месяц крупно. Интервалы помогают
модели различить эти паттерны.

In [ ]:
def order_interval_features(history):
    order_days = (
        history
        .filter(pl.col("to_ord") > 0)
        .select(["user_id", "event_date"])
        .sort(["user_id", "event_date"])
        .with_columns(
            pl.col("event_date")
            .shift(1)
            .over("user_id")
            .alias("previous_order_date")
        )
        .with_columns(
            (
                pl.col("event_date") - pl.col("previous_order_date")
            )
            .dt.total_days()
            .cast(pl.Float32)
            .alias("order_interval")
        )
    )

    return (
        order_days
        .group_by("user_id")
        .agg(
            pl.col("order_interval").mean().alias("order_interval_mean"),
            pl.col("order_interval").median().alias("order_interval_median"),
            pl.col("order_interval").std().fill_null(0).alias("order_interval_std"),
            pl.col("order_interval").min().fill_null(0).alias("order_interval_min"),
            pl.col("order_interval").max().fill_null(0).alias("order_interval_max"),
            pl.col("order_interval").tail(3).mean().fill_null(0).alias("recent_order_interval_mean"),
        )
    )


def generate_features(data, anchor_dates, user_ids):
    max_back = max(window[1] for window in WINDOWS)
    data_batch = data.filter(pl.col("user_id").is_in(user_ids))
    parts = []

    for anchor in anchor_dates:
        history = data_batch.filter(
            pl.col("event_date") <= anchor
        )

        if history.height == 0:
            continue

        known_users = history.select("user_id").unique()

        recent = history.filter(
            pl.col("event_date") >= anchor - timedelta(days=max_back)
        )

        recent_features = (
            recent
            .group_by("user_id")
            .agg(window_agg_exprs(anchor, WINDOWS))
        )

        history_features = (
            history
            .group_by("user_id")
            .agg(history_agg_exprs())
        )

        interval_features = order_interval_features(history)

        features = (
            known_users
            .join(recent_features, on="user_id", how="left")
            .join(history_features, on="user_id", how="left")
            .join(interval_features, on="user_id", how="left")
        )

        features = features.with_columns(
            (
                pl.lit(anchor).cast(pl.Date)
                - pl.col("last_activity_date")
            ).dt.total_days().cast(pl.Float32).alias("days_since_last_activity"),

            (
                pl.lit(anchor).cast(pl.Date)
                - pl.col("last_order_date")
            ).dt.total_days().cast(pl.Float32).alias("days_since_last_order"),

            (
                pl.lit(anchor).cast(pl.Date)
                - pl.col("last_cart_date")
            ).dt.total_days().cast(pl.Float32).alias("days_since_last_cart"),

            (
                pl.lit(anchor).cast(pl.Date)
                - pl.col("last_search_date")
            ).dt.total_days().cast(pl.Float32).alias("days_since_last_search"),

            (
                pl.lit(anchor).cast(pl.Date)
                - pl.col("first_activity_date")
            ).dt.total_days().cast(pl.Float32).alias("tenure_days"),

            pl.lit(anchor.month).cast(pl.Int8).alias("anchor_month"),
            pl.lit(anchor.weekday()).cast(pl.Int8).alias("anchor_weekday"),
        )

        features = features.with_columns(
            (
                pl.col("gmv_30d") /
                (pl.col("to_ord_30d") + 1e-6)
            ).alias("aov_30d"),

            (
                pl.col("gmv_90d") /
                (pl.col("to_ord_90d") + 1e-6)
            ).alias("aov_90d"),

            (
                pl.col("gmv_30d") /
                (pl.col("active_days_30d") + 1e-6)
            ).alias("gmv_per_active_day_30d"),

            (
                pl.col("to_ord_30d") /
                (pl.col("active_days_30d") + 1e-6)
            ).alias("orders_per_active_day_30d"),

            (
                pl.col("to_ord_90d") /
                (pl.col("active_days_90d") + 1e-6)
            ).alias("orders_per_active_day_90d"),

            (
                pl.col("to_ord_30d") /
                (pl.col("searches_30d") + 1e-6)
            ).alias("order_search_rate_30d"),

            (
                pl.col("to_ord_30d") /
                (pl.col("to_cart_30d") + 1e-6)
            ).alias("order_cart_rate_30d"),

            (
                pl.col("gmv_search_30d") /
                (pl.col("gmv_30d") + 1e-6)
            ).alias("search_gmv_share_30d"),

            (
                pl.col("gmv_cat_30d") /
                (pl.col("gmv_30d") + 1e-6)
            ).alias("cat_gmv_share_30d"),

            (
                pl.col("gmv_30d") /
                (pl.col("lifetime_gmv") + 1.0)
            ).alias("gmv_30_lifetime_share"),

            (
                pl.col("gmv_7d") /
                (pl.col("lifetime_gmv") + 1.0)
            ).alias("gmv_7_lifetime_share"),

            (
                pl.col("to_ord_30d") /
                (pl.col("lifetime_to_ord") + 1.0)
            ).alias("orders_30_lifetime_share"),

            (
                pl.col("active_days_30d") / 30.0
            ).alias("active_ratio_30d"),

            (
                pl.col("active_days_7d") / 7.0
            ).alias("active_ratio_7d"),

            (
                pl.col("lifetime_active_days") /
                (pl.col("tenure_days") + 1.0)
            ).alias("lifetime_activity_rate"),

            (
                pl.col("lifetime_to_ord") /
                (pl.col("tenure_days") + 1.0)
            ).alias("lifetime_order_rate"),

            (
                pl.col("gmv_7d") * 30.0 / 7.0
                - pl.col("gmv_30d")
            ).alias("gmv_7_30_accel"),

            (
                pl.col("to_ord_7d") * 30.0 / 7.0
                - pl.col("to_ord_30d")
            ).alias("orders_7_30_accel"),

            (
                pl.col("gmv_30d")
                - pl.col("gmv_60d") + pl.col("gmv_30d")
            ).alias("gmv_30_vs_previous_30d"),

            (
                pl.col("to_ord_30d")
                - pl.col("to_ord_60d") + pl.col("to_ord_30d")
            ).alias("orders_30_vs_previous_30d"),

            (
                pl.col("to_cart_30d")
                - pl.col("to_cart_60d") + pl.col("to_cart_30d")
            ).alias("cart_30_vs_previous_30d"),

            (
                pl.col("gmv_14d") * 30.0 / 14.0
                - pl.col("gmv_30d")
            ).alias("gmv_14_30_accel"),

            (
                pl.col("to_ord_14d") * 30.0 / 14.0
                - pl.col("to_ord_30d")
            ).alias("orders_14_30_accel"),

            (
                pl.col("to_cart_14d") * 30.0 / 14.0
                - pl.col("to_cart_30d")
            ).alias("cart_14_30_accel"),

            (
                pl.col("gmv_7d") /
                (pl.col("gmv_14d") / 2.0 + 1.0)
            ).alias("gmv_7_vs_14_ratio"),

            (
                pl.col("to_ord_7d") /
                (pl.col("to_ord_14d") / 2.0 + 1.0)
            ).alias("orders_7_vs_14_ratio"),

            (
                pl.col("to_ord_30d") /
                (pl.col("to_ord_90d") / 3.0 + 1.0)
            ).alias("orders_30_vs_90_ratio"),

            (
                pl.col("gmv_30d") /
                (pl.col("gmv_90d") / 3.0 + 1.0)
            ).alias("gmv_30_vs_90_ratio"),

            (
                pl.col("to_ord_7d") > 0
            ).cast(pl.Float32).alias("has_order_7d"),

            (
                pl.col("to_ord_14d") > 0
            ).cast(pl.Float32).alias("has_order_14d"),

            (
                pl.col("to_ord_30d") > 0
            ).cast(pl.Float32).alias("has_order_30d"),

            (
                pl.col("to_cart_7d") > 0
            ).cast(pl.Float32).alias("has_cart_7d"),

            (
                pl.col("to_cart_30d") > 0
            ).cast(pl.Float32).alias("has_cart_30d"),

            (
                pl.col("gmv_7d") > 0
            ).cast(pl.Float32).alias("has_gmv_7d"),

            (
                pl.col("gmv_30d") > 0
            ).cast(pl.Float32).alias("has_gmv_30d"),

            (
                pl.col("searches_7d") > 0
            ).cast(pl.Float32).alias("has_search_7d"),

            (
                pl.col("searches_30d") > 0
            ).cast(pl.Float32).alias("has_search_30d"),

            (
                pl.col("days_since_last_order") /
                (pl.col("order_interval_median") + 1.0)
            ).alias("recency_vs_order_cycle"),

            (
                pl.col("order_interval_median") /
                (pl.col("tenure_days") + 1.0)
            ).alias("cycle_vs_tenure"),

            (
                pl.col("to_ord_30d") /
                (pl.col("active_days_30d") + 1.0)
            ).alias("order_day_share_30d"),

            (
                pl.col("to_ord_90d") /
                (pl.col("active_days_90d") + 1.0)
            ).alias("order_day_share_90d"),
        )

        log_cols = [
            "gmv_7d",
            "gmv_14d",
            "gmv_30d",
            "gmv_60d",
            "gmv_90d",
            "to_ord_7d",
            "to_ord_14d",
            "to_ord_30d",
            "to_ord_90d",
            "to_cart_30d",
            "searches_30d",
            "lifetime_gmv",
            "lifetime_to_ord",
            "order_interval_mean",
            "order_interval_median",
            "days_since_last_order",
            "days_since_last_activity",
        ]

        features = features.with_columns([
            pl.col(c)
            .clip(lower_bound=0)
            .log1p()
            .alias(f"log1p_{c}")
            for c in log_cols
            if c in features.columns
        ])

        date_cols = {
            "user_id",
            "first_activity_date",
            "last_activity_date",
            "last_order_date",
            "last_cart_date",
            "last_search_date",
        }

        fill_cols = [
            c for c in features.columns
            if c not in date_cols
        ]

        features = features.with_columns([
            pl.col(c).fill_null(0.0)
            for c in fill_cols
        ])

        features = features.with_columns(
            pl.col("days_since_last_activity").fill_null(9999.0),
            pl.col("days_since_last_order").fill_null(9999.0),
            pl.col("days_since_last_cart").fill_null(9999.0),
            pl.col("days_since_last_search").fill_null(9999.0),
            pl.col("tenure_days").fill_null(0.0),
            pl.lit(anchor).cast(pl.Date).alias("anchor_date"),
        )

        parts.append(features)

        del history, recent, recent_features, history_features, interval_features

    if not parts:
        return pl.DataFrame()

    return pl.concat(parts, how="diagonal_relaxed")


## Формирование target

Для каждого anchor берём окно `[anchor + 1, anchor + 30]` и суммируем `gmv`
по каждому пользователю. Если покупок не было — target равен 0.

Это ровно та величина, которую нужно предсказывать в соревновании.

In [ ]:
def generate_targets(data, anchor_dates, user_ids, horizon=30):
    data_batch = data.filter(pl.col("user_id").is_in(user_ids))
    parts = []

    for anchor in anchor_dates:
        target = (
            data_batch
            .filter(
                pl.col("event_date").is_between(
                    anchor + timedelta(days=1),
                    anchor + timedelta(days=horizon)
                )
            )
            .group_by("user_id")
            .agg(
                pl.col("gmv").sum().alias("target")
            )
            .with_columns(
                pl.lit(anchor).cast(pl.Date).alias("anchor_date")
            )
        )

        parts.append(target)

    if not parts:
        return pl.DataFrame()

    return (
        pl.concat(parts, how="diagonal_relaxed")
        .with_columns(
            pl.col("target").fill_null(0.0)
        )
    )


## Построение временных фолдов

Для каждого anchor создаём признаки и target, соединяем их по
`(user_id, anchor_date)` и сохраняем в parquet.

Parquet-кэш позволяет не пересчитывать признаки перед каждым экспериментом
и обучать LightGBM быстрее.

In [ ]:
for fold_idx, anchor in enumerate(anchors):
    fold_dir = FEATURES_DIR / f"fold_{fold_idx:02d}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    for batch_idx in range(n_batches):
        batch_users = user_ids[
            batch_idx * BATCH_SIZE:
            (batch_idx + 1) * BATCH_SIZE
        ]

        features = generate_features(
            data,
            [anchor],
            batch_users
        )

        targets = generate_targets(
            data,
            [anchor],
            batch_users,
            HORIZON
        )

        fold = (
            features
            .join(
                targets,
                on=["anchor_date", "user_id"],
                how="left"
            )
            .with_columns(
                pl.col("target").fill_null(0.0)
            )
        )

        fold.write_parquet(
            fold_dir / f"batch_{batch_idx:04d}.parquet",
            compression="zstd"
        )

        del features, targets, fold
        gc.collect()

    print(f"Fold {fold_idx}: {anchor}")


Fold 0: 2025-12-03
Fold 1: 2025-12-17
Fold 2: 2025-12-31
Fold 3: 2026-01-14


## Финальные признаки

Отдельно строим признаки на последнюю дату — **2026-02-13**.
Target для них не создаётся, так как будущее нам неизвестно.

Именно эти признаки пойдут в финальную модель для прогноза периода
с 14 февраля по 15 марта 2026 года.

In [ ]:
final_dir = FEATURES_DIR / "fold_end"
final_dir.mkdir(parents=True, exist_ok=True)

for batch_idx in range(n_batches):
    batch_users = user_ids[
        batch_idx * BATCH_SIZE:
        (batch_idx + 1) * BATCH_SIZE
    ]

    features = generate_features(
        data,
        [anchor_end],
        batch_users
    )

    features.write_parquet(
        final_dir / f"batch_{batch_idx:04d}.parquet",
        compression="zstd"
    )

    del features
    gc.collect()

print(f"Final anchor: {anchor_end}")


Final anchor: 2026-02-13


## Подготовка признаков

Читаем один батч, чтобы получить список колонок, и исключаем из него:
- `user_id` и `anchor_date`;
- `target`;
- все даты (`first_activity_date`, `last_order_date` и т.д.).

Остальные колонки — числовые признаки, которые пойдут в LightGBM.

In [ ]:
sample = pl.read_parquet(
    sorted((FEATURES_DIR / "fold_00").glob("batch_*.parquet"))[0]
)

DROP_COLS = [
    "user_id",
    "anchor_date",
    "target",
    "first_activity_date",
    "last_activity_date",
    "last_order_date",
    "last_cart_date",
    "last_search_date",
]

feature_cols = [
    c for c in sample.columns
    if c not in DROP_COLS
]

print(f"Features: {len(feature_cols)}")


Features: 187


## Загрузка фолдов

Собираем все батчи одного фолда в единые numpy-массивы `X` и `y`.
Используем `float32`, чтобы сэкономить память при обучении.

In [ ]:
def load_fold_arrays(fold_name):
    paths = sorted(
        (FEATURES_DIR / fold_name).glob("batch_*.parquet")
    )

    X_parts = []
    y_parts = []

    for path in paths:
        df = pl.read_parquet(path)

        X_parts.append(
            df.select(feature_cols)
            .to_numpy()
            .astype(np.float32, copy=False)
        )

        y_parts.append(
            df["target"]
            .to_numpy()
            .astype(np.float32, copy=False)
        )

        del df

    X = np.concatenate(X_parts, axis=0)
    y = np.concatenate(y_parts, axis=0)

    del X_parts, y_parts
    gc.collect()

    return X, y


## Кросс-валидация LightGBM

Схема обучения такая же, как в CatBoost-ноутбуке:
- обучаемся на всех предыдущих фолдах;
- валидируемся на следующем;
- target — `log1p(GMV)`;
- метрика — RMSE в лог-пространстве, то есть RMSLE.

Параметры LightGBM:
- `num_leaves=31`, `learning_rate=0.03`;
- `subsample=0.85`, `colsample_bytree=0.85`;
- `min_child_samples=80`, `reg_alpha=0.1`, `reg_lambda=1.0`.

Early stopping — 150 раундов, максимум 3000 итераций.

In [ ]:
cv_results = []
cv_models = []
best_iterations = []

params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 80,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "verbosity": -1,
    "seed": 42,
    "num_threads": 4,
}

for valid_idx in range(1, N_FOLDS):
    train_X_parts = []
    train_y_parts = []

    for train_idx in range(valid_idx):
        X_part, y_part = load_fold_arrays(
            f"fold_{train_idx:02d}"
        )
        train_X_parts.append(X_part)
        train_y_parts.append(y_part)

    X_train = np.concatenate(train_X_parts, axis=0)
    y_train = np.concatenate(train_y_parts, axis=0)

    del train_X_parts, train_y_parts

    X_valid, y_valid = load_fold_arrays(
        f"fold_{valid_idx:02d}"
    )

    y_train_log = np.log1p(
        np.clip(y_train, 0, None)
    )

    y_valid_log = np.log1p(
        np.clip(y_valid, 0, None)
    )

    train_set = lgb.Dataset(
        X_train,
        label=y_train_log,
        free_raw_data=False,
    )

    valid_set = lgb.Dataset(
        X_valid,
        label=y_valid_log,
        reference=train_set,
        free_raw_data=False,
    )

    model = lgb.train(
        params,
        train_set,
        num_boost_round=3000,
        valid_sets=[valid_set],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(150, verbose=True),
            lgb.log_evaluation(200),
        ],
    )

    pred_log = model.predict(
        X_valid,
        num_iteration=model.best_iteration
    )

    pred = np.expm1(pred_log)
    pred = np.clip(pred, 0, None)

    score = rmsle(
        y_valid,
        pred
    )

    cv_results.append({
        "fold": valid_idx,
        "rmsle": score,
        "best_iteration": model.best_iteration,
    })

    best_iterations.append(model.best_iteration)
    cv_models.append(model)

    print(
        f"Fold {valid_idx}: "
        f"RMSLE={score:.6f}, "
        f"best_iteration={model.best_iteration}"
    )

    del train_set, valid_set
    del X_train, y_train, y_train_log
    del X_valid, y_valid, y_valid_log
    gc.collect()

cv_results_df = pl.DataFrame(cv_results)

print(cv_results_df)
print(f"Mean RMSLE: {cv_results_df['rmsle'].mean():.6f}")
print(f"Median RMSLE: {cv_results_df['rmsle'].median():.6f}")


Training until validation scores don't improve for 150 rounds
[200]	valid's rmse: 1.74541
[400]	valid's rmse: 1.74403
[600]	valid's rmse: 1.74315
[800]	valid's rmse: 1.74238
[1000]	valid's rmse: 1.74139
[1200]	valid's rmse: 1.74065
[1400]	valid's rmse: 1.74005
[1600]	valid's rmse: 1.73936
[1800]	valid's rmse: 1.73871
[2000]	valid's rmse: 1.73812
[2200]	valid's rmse: 1.73772
[2400]	valid's rmse: 1.73718
[2600]	valid's rmse: 1.73666
[2800]	valid's rmse: 1.73605
[3000]	valid's rmse: 1.73553
Did not meet early stopping. Best iteration is:
[2999]	valid's rmse: 1.73553
Fold 1: RMSLE=1.735527, best_iteration=2999
Training until validation scores don't improve for 150 rounds
[200]	valid's rmse: 1.70842
[400]	valid's rmse: 1.70579
[600]	valid's rmse: 1.70481
[800]	valid's rmse: 1.70411
[1000]	valid's rmse: 1.7034
[1200]	valid's rmse: 1.70288
[1400]	valid's rmse: 1.70242
[1600]	valid's rmse: 1.702
[1800]	valid's rmse: 1.70166
[2000]	valid's rmse: 1.70127
[2200]	valid's rmse: 1.701
[2400]	valid's

## Feature importance

In [ ]:
importance = cv_models[-1].feature_importance(
    importance_type="gain"
)

feature_importance = (
    pl.DataFrame({
        "feature": feature_cols,
        "importance": importance,
    })
    .sort("importance", descending=True)
)

print(feature_importance.head(30))


shape: (30, 2)
┌────────────────────────────┬──────────────┐
│ feature                    ┆ importance   │
│ ---                        ┆ ---          │
│ str                        ┆ f64          │
╞════════════════════════════╪══════════════╡
│ lifetime_order_rate        ┆ 7.0358e6     │
│ lifetime_to_ord            ┆ 6.7542e6     │
│ to_ord_90d                 ┆ 4.4958e6     │
│ has_search_to_ord_90d      ┆ 2.4868e6     │
│ log1p_lifetime_to_ord      ┆ 1.2768e6     │
│ …                          ┆ …            │
│ recent_order_interval_mean ┆ 82167.338049 │
│ searches_14d               ┆ 81402.648049 │
│ log1p_gmv_90d              ┆ 81314.151565 │
│ log1p_gmv_60d              ┆ 75916.616888 │
│ days_since_last_activity   ┆ 72320.032713 │
└────────────────────────────┴──────────────┘


## Финальное обучение

Берём медиану лучших итераций по фолдам и обучаем одну финальную модель
на всех доступных данных.

Модель сохраняем в `data/v4_lgbm/models/lgbm_ltv_final.txt`,
чтобы её можно было переиспользовать без повторного обучения.

In [ ]:
final_iterations = int(
    np.median(
        np.asarray(best_iterations)
    )
)

print(f"Final iterations: {final_iterations}")

train_X_parts = []
train_y_parts = []

for fold_idx in range(N_FOLDS):
    X_part, y_part = load_fold_arrays(
        f"fold_{fold_idx:02d}"
    )
    train_X_parts.append(X_part)
    train_y_parts.append(y_part)

X_train_all = np.concatenate(
    train_X_parts,
    axis=0
)

y_train_all = np.concatenate(
    train_y_parts,
    axis=0
)

del train_X_parts, train_y_parts
gc.collect()

y_train_all_log = np.log1p(
    np.clip(y_train_all, 0, None)
)

train_set = lgb.Dataset(
    X_train_all,
    label=y_train_all_log,
    free_raw_data=False,
)

final_model = lgb.train(
    params,
    train_set,
    num_boost_round=final_iterations,
)

FINAL_MODEL_PATH = MODELS_DIR / "lgbm_ltv_final.txt"

final_model.save_model(
    str(FINAL_MODEL_PATH)
)

print(f"Saved: {FINAL_MODEL_PATH}")


Final iterations: 2999
Saved: /content/drive/MyDrive/Colab Notebooks/e-cup/data/v4_lgbm/models/lgbm_ltv_final.txt


## Финальный прогноз

Применяем финальную модель к признакам на 2026-02-13.
Предсказания переводим из лог-пространства через `expm1`
и обрезаем снизу нулём.

Собираем таблицу `user_id, predict` для всех 250 000 пользователей.

In [ ]:
final_paths = sorted(
    (FEATURES_DIR / "fold_end").glob("batch_*.parquet")
)

prediction_parts = []

for path in final_paths:
    df = pl.read_parquet(path)

    X_batch = (
        df.select(feature_cols)
        .to_numpy()
        .astype(np.float32, copy=False)
    )

    pred_log = final_model.predict(
        X_batch
    )

    pred = np.expm1(pred_log)
    pred = np.clip(pred, 0, None)

    prediction_parts.append(
        pl.DataFrame({
            "user_id": df["user_id"],
            "predict": pred,
        })
    )

    del df, X_batch, pred_log, pred
    gc.collect()

submission = (
    pl.concat(
        prediction_parts,
        how="vertical"
    )
    .select(["user_id", "predict"])
    .sort("user_id")
)

del prediction_parts
gc.collect()

print(f"Rows: {submission.height:,}")
print(f"Unique users: {submission['user_id'].n_unique():,}")
print(f"Nulls: {submission['predict'].null_count()}")
print(f"Min: {submission['predict'].min():.6f}")
print(f"Median: {submission['predict'].median():.6f}")
print(f"Mean: {submission['predict'].mean():.6f}")
print(f"Max: {submission['predict'].max():.6f}")


Rows: 250,000
Unique users: 250,000
Nulls: 0
Min: 0.000000
Median: 6.408610
Mean: 35.713317
Max: 4418.844864


## Сохранение результата

In [ ]:
submission.write_csv(
    SUBMISSION_PATH
)

print(f"Saved: {SUBMISSION_PATH}")


Saved: /content/drive/MyDrive/Colab Notebooks/e-cup/data/submission_lgbm_v1.csv


## Скачать CSV

In [ ]:
from google.colab import files

files.download(
    str(SUBMISSION_PATH)
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Скачать модель

In [ ]:
from google.colab import files

files.download(
    str(FINAL_MODEL_PATH)
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>